In [2]:
# Cell 0 — install dependencies (run once)
%pip install timm

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [3]:
# Cell 1
%load_ext autoreload
%autoreload 2
import os
import pandas as pd
import config
import step1_data
from train import fit

In [4]:
# Cell 2
class_to_images, class_names = step1_data.scan_dataset(config.DATA_DIR)
train_split, val_split, test_split = step1_data.split_dataset(
    class_to_images, class_names,
    config.TRAIN_RATIO, config.VAL_RATIO, config.RANDOM_SEED,
)
cols = ["filepath", "label_idx"]
train_df = pd.DataFrame(train_split, columns=cols)
val_df   = pd.DataFrame(val_split,   columns=cols)
test_df  = pd.DataFrame(test_split,  columns=cols)
num_classes = len(class_names)
print(f"train={len(train_df)}  val={len(val_df)}  test={len(test_df)}  classes={num_classes}")

train=3331  val=416  test=418  classes=2


In [8]:
# Cell 3
config.HEAD_EPOCHS = 1
config.FINETUNE_EPOCHS = 1
model, history = fit(train_df, val_df, test_df, num_classes)

[Stage 1] Head only: 3,074 / 10,699,306 trainable
[S1 1/1] train 1.2043/0.7022 | val 1.1679/0.6707 | 341s
[Stage 2] Full network: 10,699,306 / 10,699,306 trainable
[S2 1/1] train 0.9546/0.7139 | val 0.8141/0.7115 | 1092s

Best val acc: 0.7115 (saved to outputs\best_model.pth)
Test: 0.7258 loss / 0.7153 acc


In [5]:
# Cell 4 — evaluation
from evaluate import evaluate_model

report_df, cm = evaluate_model(train_df, val_df, test_df, class_names)
report_df.head(10)

  ✓ classification_report.csv

              precision    recall  f1-score   support

        test       0.21      0.16      0.18        82
       train       0.81      0.85      0.83       336

    accuracy                           0.72       418
   macro avg       0.51      0.50      0.50       418
weighted avg       0.69      0.72      0.70       418

  ✓ confusion_matrix.png

  Overall test accuracy: 0.7153


,precision,recall,f1-score,support
test,0.206349,0.158537,0.179310,82.000000
train,0.805634,0.851190,0.827786,336.000000
accuracy,0.715311,0.715311,0.715311,0.715311
macro avg,0.505992,0.504864,0.503548,418.000000
weighted avg,0.688071,0.715311,0.700573,418.000000
